In [1]:
import random
import numpy as np
import networkx as nx
import networkit as nk
from typing import Union
from collections import deque
from littleballoffur.sampler import Sampler
import torch
import torch_geometric.transforms as T
from torch_geometric.datasets.dblp import DBLP
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx

NKGraph = type(nk.graph.Graph())
NXGraph = nx.classes.graph.Graph

Small graphs are obtained from real datasets by modifying the ForestFireSampler code in https://little-ball-of-fur.readthedocs.io/en/latest/_modules/littleballoffur/exploration_sampling/forestfiresampler.html

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import csv
import random
import os

from littleballoffur import ForestFireSampler
import time

In [3]:
class ForestFireSampler(Sampler):
    r"""An implementation of forest fire sampling. The procedure is a stochastic
    snowball sampling method where the expansion is proportional to the burning probability.
    `"For details about the algorithm see this paper." <https://cs.stanford.edu/people/jure/pubs/sampling-kdd06.pdf>`_


    Args:
        number_of_nodes (int): Number of sampled nodes. Default is 100.
        p (float): Burning probability. Default is 0.4.
        seed (int): Random seed. Default is 42.
    """

    def __init__(
            self,
            number_of_nodes: int = 100,
            p: float = 0.4,
            seed: int = 42,
            max_visited_nodes_backlog: int = 100,
            restart_hop_size: int = 10,
    ):
        self.number_of_nodes = number_of_nodes
        self.p = p
        #self.seed = seed
        #self._set_seed()
        self.restart_hop_size = restart_hop_size
        self.max_visited_nodes_backlog = max_visited_nodes_backlog

    def _create_node_sets(self, graph):
        """
        Create a starting set of nodes.
        """
        self._sampled_nodes = set()
        self._set_of_nodes = set(range(self.backend.get_number_of_nodes(graph)))
        self._visited_nodes = deque(maxlen=self.max_visited_nodes_backlog)

    def _start_a_fire(self, graph):
        """
        Starting a forest fire from a single node.
        """
        remaining_nodes = list(self._set_of_nodes.difference(self._sampled_nodes))
        #seed_node = random.choice(remaining_nodes)
        #modified code to select start node from the list of Author nodes
        author_nodes= [x for x, y in undirected_dblp.nodes(data=True) if y['type'] == 'author']
        seed_node = random.choice(author_nodes)
        self._sampled_nodes.add(seed_node)
        node_queue = deque([seed_node])
        while len(self._sampled_nodes) < self.number_of_nodes:
            if len(node_queue) == 0:
                node_queue = deque(
                    [
                        self._visited_nodes.popleft()
                        for k in range(
                        min(self.restart_hop_size, len(self._visited_nodes))
                    )
                    ]
                )
                if len(node_queue) == 0:
                    print(
                        "Warning: could not collect the required number of nodes. The fire could not find enough nodes to burn."
                    )
                    break
            top_node = node_queue.popleft()
            self._sampled_nodes.add(top_node)
            neighbors = set(self.backend.get_neighbors(graph, top_node))
            unvisited_neighbors = neighbors.difference(self._sampled_nodes)
            score = np.random.geometric(self.p)
            count = min(len(unvisited_neighbors), score)
            burned_neighbors = random.sample(unvisited_neighbors, count)
            self._visited_nodes.extendleft(
                unvisited_neighbors.difference(set(burned_neighbors))
            )
            for neighbor in burned_neighbors:
                if len(self._sampled_nodes) >= self.number_of_nodes:
                    break
                node_queue.extend([neighbor])

    def sample(self, graph: Union[NXGraph, NKGraph]) -> Union[NXGraph, NKGraph]:
        """
        Sampling nodes iteratively with a forest fire sampler.

        Arg types:
            * **graph** *(NetworkX or NetworKit graph)* - The graph to be sampled from.

        Return types:
            * **new_graph** *(NetworkX or NetworKit graph)* - The graph of sampled nodes.
        """
        self._deploy_backend(graph)
        self._check_number_of_nodes(graph)
        self._create_node_sets(graph)
        while len(self._sampled_nodes) < self.number_of_nodes:
            self._start_a_fire(graph)
        new_graph = self.backend.get_subgraph(graph, self._sampled_nodes)
        return new_graph

In [4]:
def plot_graph(G,attribute):

    color_class_map = {0: 'blue', 1: 'red', 2: 'darkgreen', 3: 'orange'}

    nx.draw(G, 
        with_labels=True, node_color=[color_class_map[node[1][attribute]] 
                        for node in G.nodes(data=True)], 
            node_size=200,
        font_color='black')
    plt.show() 

DBLP

In [5]:
dataset = DBLP(root='./dblp_data', transform=T.Constant(node_types='conference'))
data = dataset[0]

Extracting dblp_data/raw/DBLP_processed.zip
Processing...
Done!


In [6]:
def get_node_types(G):
    for n in G.nodes:
        if G.nodes[n]['type'] == 'author':
            G.nodes[n]['type'] = 1
        elif G.nodes[n]['type'] == 'paper':
            G.nodes[n]['type'] = 0
        elif G.nodes[n]['type'] == 'term':
            G.nodes[n]['type'] = 2
        elif G.nodes[n]['type'] == 'conference':
            G.nodes[n]['type'] = 3

In [7]:
def feature_selection(df_sampled):
    col_sum = df_sampled.sum(axis=0)
    sorted_colsum = sorted(col_sum, reverse=True)
    index_list = []
    for i in sorted_colsum[:50]:
        index_list.append(list(col_sum).index(i))
    imp_feat = df_sampled[index_list]
    return imp_feat

In [8]:
#Original author node features
author = data['author'].x.tolist()
author_df = pd.DataFrame(author)
author_df['class'] = data['author'].y.tolist()
author_df

,0,1,2,3,4,5,6,7,8,9,...,325,326,327,328,329,330,331,332,333,class
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4052,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4053,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4054,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4055,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [9]:
author_class0 = author_df[author_df['class'] == 0].drop(['class'], axis = 1)
author_class0 = author_class0.reset_index(drop=True)
author_class0 = feature_selection(author_class0)
author_class0

,63,64,295,222,65,221,23,296,162,88,...,22,22,251,9,137,137,124,13,13,44
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1192,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1193,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1194,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
author_class1 = author_df[author_df['class'] == 1].drop(['class'], axis = 1)
author_class1= author_class1.reset_index(drop=True)
author_class1 = feature_selection(author_class1)
author_class1

,63,170,321,23,39,152,88,35,15,65,...,77,77,77,302,80,80,185,314,64,64
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
740,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
741,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
742,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
743,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
author_class2 = author_df[author_df['class'] == 2].drop(['class'], axis = 1)
author_class2= author_class2.reset_index(drop=True)
author_class2 = feature_selection(author_class2)
author_class2

,152,23,321,148,228,295,15,15,296,11,...,83,72,188,21,21,21,21,21,173,60
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1104,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1105,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1106,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1107,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
author_class3 = author_df[author_df['class'] == 3].drop(['class'], axis = 1)
author_class3= author_class3.reset_index(drop=True)
author_class3 = feature_selection(author_class3)
author_class3

,242,135,23,321,253,329,302,222,81,295,...,88,206,206,101,106,106,106,151,151,97
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0
3,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1001,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1002,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1003,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
#Original paper node features
paper = data['paper'].x.tolist()
paper_df = pd.DataFrame(paper)
paper_df= paper_df.reset_index(drop=True)
paper_df = feature_selection(paper_df)
paper_df

,11,31,950,955,3074,4065,2081,2358,2338,1866,...,3691,3703,3969,1330,2161,3444,1201,1057,2967,3430
0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14323,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14324,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14325,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14326,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
#Original paper node features
term = data['term'].x.tolist()
term_df = pd.DataFrame(term)
term_df = term_df.reset_index(drop=True)
term_df

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,-0.69241,-0.465920,1.154000,0.066483,-0.676250,0.693760,0.53680,0.206780,0.12607,-0.36763,...,1.32870,0.074684,-0.34136,0.873600,-0.065725,-0.780720,0.902000,0.917840,0.19945,-0.63601
1,1.20310,-0.400280,0.073991,1.041500,0.051753,0.411660,-0.98656,-0.794660,0.36033,0.54428,...,0.38339,-0.572190,-0.16915,0.139840,-0.774300,-0.061819,0.218870,1.326200,-0.33245,0.81980
2,0.37481,0.573140,0.480170,-0.056679,0.704740,0.715910,0.32591,-0.099446,0.51029,0.64193,...,0.74200,-0.163610,0.44295,0.400600,-0.235400,-0.276530,-0.203020,1.152200,0.60099,-0.43094
3,0.54148,-0.419840,-0.334260,-0.412010,0.112200,1.047600,0.47449,-0.751110,0.40817,0.74477,...,0.45971,-0.064420,-0.27582,1.021800,-0.037676,-0.405910,1.231400,0.472220,0.69127,0.37195
4,0.16408,0.927990,-1.573400,-0.859870,0.369720,0.852800,0.20269,-0.620400,-1.00010,0.29585,...,0.66077,-0.952820,-0.36583,1.758900,-0.497820,0.599950,0.756170,1.384800,-0.11633,-0.62254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7718,0.41800,0.249680,-0.412420,0.121700,0.345270,-0.044457,-0.49688,-0.178620,-0.00066,-0.65660,...,-0.29871,-0.157490,-0.34758,-0.045637,-0.442510,0.187850,0.002785,-0.184110,-0.11514,-0.78581
7719,-1.01450,1.239400,-1.618500,-1.033600,-0.056811,0.270890,-0.38817,-0.353350,-0.51784,1.10460,...,0.97806,0.158320,-0.54189,-0.186700,-0.334690,0.391300,0.524110,-0.059546,-0.78882,-0.71343
7720,0.41800,0.249680,-0.412420,0.121700,0.345270,-0.044457,-0.49688,-0.178620,-0.00066,-0.65660,...,-0.29871,-0.157490,-0.34758,-0.045637,-0.442510,0.187850,0.002785,-0.184110,-0.11514,-0.78581
7721,0.17239,-0.272270,-1.336800,1.170600,-0.621270,-0.528470,1.32210,0.145990,-0.15398,-1.24660,...,-0.76113,-0.583970,-0.13828,0.907620,-0.114690,-0.746190,0.723290,-0.088101,0.02252,0.11660


In [15]:
def remap_indices(G):
    val_list = [*range(0, G.number_of_nodes(), 1)]
    return dict(zip(G,val_list)) 

In [16]:
def add_node_features(G):
    for n in G.nodes:
        node_type = G.nodes[n]["type"] 
        
        if node_type == 0:
            node_id = n
            paper_node_feature = paper_df.loc[int(node_id), :].values.flatten().tolist()
            G.nodes[n]["feature"] = paper_node_feature 

        elif node_type == 1:
            node_id = n
            node_class = G.nodes[n]["class"]
            
            if node_class == 0:
                author_node_feature = author_class0.loc[int(node_id), :].values.flatten().tolist()
                G.nodes[n]["feature"] = author_node_feature 
                G.nodes[n]["class"] = node_class

            elif node_class == 1:
                author_node_feature = author_class1.loc[int(node_id), :].values.flatten().tolist()
                G.nodes[n]["feature"] = author_node_feature 
                G.nodes[n]["class"] = node_class

            elif node_class == 2:                
                author_node_feature = author_class2.loc[int(node_id), :].values.flatten().tolist()
                G.nodes[n]["feature"] = author_node_feature 
                G.nodes[n]["class"] = node_class
                
            elif node_class == 3:     
                author_node_feature = author_class3.loc[int(node_id), :].values.flatten().tolist()
                G.nodes[n]["feature"] = author_node_feature 
                G.nodes[n]["class"] = node_class
                
        elif node_type == 2:
            node_id = n
            term_node_feature = term_df.loc[int(node_id), :].values.flatten().tolist()
            G.nodes[n]["feature"] = term_node_feature 
        elif node_type == 3:
            node_id = n
            term_node_feature = np.zeros(50)
            G.nodes[n]["feature"] = term_node_feature

    return G

In [17]:
networkX_dblp_graph = to_networkx(data)
undirected_dblp = networkX_dblp_graph.to_undirected()
nodes_attr = author_df[['class']].set_index(author_df.index).to_dict(orient = 'index')
nx.set_node_attributes(undirected_dblp, nodes_attr)
adj_list= []
node_type_list = []
node_feature_list = []
for i in range(0,200):
    model = ForestFireSampler(15)
    graph = model.sample(undirected_dblp)
    if nx.is_connected(graph):
        mapping = remap_indices(graph)
        graph = nx.relabel_nodes(graph, mapping)
        
        get_node_types(graph)
        graph = add_node_features(graph)
        
        A = nx.adjacency_matrix(graph)
        adj = torch.tensor(np.asarray(A.todense()))
        adj_list.append(adj)
        
        node_type = nx.get_node_attributes(graph, "type")
        node_feature = nx.get_node_attributes(graph, "feature")
        
        node_type_list.append(list(node_type.values()))
        node_feature_list.append(list(node_feature.values()))
        
    else:
        print('Disconnected graph')      

In [18]:
len(adj_list)

200

In [19]:
#Save graphs with node features and node types for graph generation using VAE(baseline)
torch.save([adj_list,node_type_list,node_feature_list],'real_dblp_vae_15.pt')

In [20]:
#Load graphs
adjs, node_types, node_feats = torch.load('real_dblp_vae_15.pt')

In [21]:
for i, adj in enumerate(adjs[:1]):
    print('adjacency matrix', adj)
    print('node types', node_types[i])
    print('node features', node_feats[i])

adjacency matrix tensor([[0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]])
node types [0, 0, 0, 0, 2, 2, 2, 1, 0, 2, 2, 0, 2, 1, 0]
node features [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [22]:
#Save real graphs with node types for graph generation using Diffusion Models 
torch.save([adj_list,node_type_list],'real_dblp_15.pt')

In [23]:
#Load graphs
adjs, node_types = torch.load('real_dblp_15.pt')

In [24]:
for i, adj in enumerate(adjs[:1]):
    print('adjacency matrix', adj)
    print('node types', node_types[i])

adjacency matrix tensor([[0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]])
node types [0, 0, 0, 0, 2, 2, 2, 1, 0, 2, 2, 0, 2, 1, 0]
